# Unity Catalog Governance and Access Control

## Purpose

I use this notebook to define the governance strategy for the Health Insurance
lakehouse.

The project already uses Unity Catalog for:

- catalog and schema organization
- governed data quality rules
- object ownership
- lineage through pipeline-managed datasets

In this notebook I extend governance with:

- sensitive-data classification
- attribute-based column masking
- role-based access design
- governance metadata validation

### Security model

I separate access by data sensitivity.

Silver clinical data contains detailed FHIR records and should be restricted to
engineering or privileged clinical users.

Gold aggregate claims datasets can be exposed more broadly to analytical users.

The Gold `patient_clinical_summary` contains direct patient attributes, so I
classify sensitive columns with governed tags and apply a Unity Catalog ABAC
column-mask policy.

I do not hard-code individual user identities in the project. Access is designed
around account groups so the same policy can be reused across environments.

In [0]:
%sql
-- confirming the catalog and schemas used by the governance model.

SHOW SCHEMAS IN health_insurance;

## Governed PII classification

I classify direct patient identifiers using a governed `pii` tag.

I use controlled values instead of arbitrary free-text tags so sensitive-data
classification remains consistent.

The tag supports the following categories:

- name
- medical_record_number
- phone
- date_of_birth
- location

The tag itself contains only classification metadata and never contains actual
patient information.

In [0]:
%sql
-- checking whether the governed PII tag already exists.

SHOW GOVERNED TAGS LIKE 'pii';

In [0]:
%sql
-- creating an account-level governed tag for PII classification.

CREATE GOVERNED TAG pii
DESCRIPTION 'Classifies personally identifiable information in healthcare datasets'
VALUES (
    'name',
    'medical_record_number',
    'phone',
    'date_of_birth',
    'location'
);

In [0]:
%sql
-- classifying direct Patient identifiers in the Gold clinical model.

SET TAG ON COLUMN
health_insurance.gold.patient_clinical_summary.given_name
`pii` = `name`;

SET TAG ON COLUMN
health_insurance.gold.patient_clinical_summary.family_name
`pii` = `name`;

SET TAG ON COLUMN
health_insurance.gold.patient_clinical_summary.medical_record_number
`pii` = `medical_record_number`;

SET TAG ON COLUMN
health_insurance.gold.patient_clinical_summary.phone
`pii` = `phone`;

SET TAG ON COLUMN
health_insurance.gold.patient_clinical_summary.birth_date
`pii` = `date_of_birth`;

SET TAG ON COLUMN
health_insurance.gold.patient_clinical_summary.postal_code
`pii` = `location`;

In [0]:
%sql
-- verifying the column-level PII classifications stored in Unity Catalog.

SELECT
    catalog_name,
    schema_name,
    table_name,
    column_name,
    tag_name,
    tag_value

FROM health_insurance.information_schema.column_tags

WHERE schema_name = 'gold'
  AND table_name = 'patient_clinical_summary'

ORDER BY column_name;

## PII masking function

I create a reusable Unity Catalog SQL function that masks sensitive string
values.

The masking function does not contain table-specific logic.

Instead, Unity Catalog ABAC decides which tagged columns receive the function.

This separation allows:

- classification to be managed through tags
- policy scope to be managed through Unity Catalog
- masking behavior to remain reusable

In [0]:
%sql
-- creating the reusable PII masking function.

CREATE OR REPLACE FUNCTION
health_insurance.governance.mask_pii(value STRING)

RETURNS STRING

DETERMINISTIC

RETURN
    CASE
        WHEN value IS NULL THEN NULL
        ELSE '***MASKED***'
    END;

## Account-group model

I design access around Unity Catalog account groups instead of individual users.

Expected groups:

`healthcare_privileged`

Users authorized to access identifiable Patient information.

`healthcare_analysts`

Users who need analytical access but should not receive direct Patient
identifiers.

The group names are environment-level configuration rather than project-owned
data. They should be created through Databricks account administration before
the access policies are activated.

In [0]:
%sql
-- masking tagged PII for analysts while exempting privileged clinical users.

CREATE OR REPLACE POLICY mask_healthcare_pii
ON SCHEMA health_insurance.gold

COLUMN MASK health_insurance.governance.mask_pii

TO `healthcare_analysts`
EXCEPT `healthcare_privileged`

FOR TABLES

MATCH COLUMNS
    has_tag('pii') AS sensitive_column

ON COLUMN sensitive_column;

## Least-privilege access

I combined ABAC with standard Unity Catalog privileges.

Analysts receive access to curated Gold datasets rather than unrestricted access
to Bronze or Silver clinical data.

Privileged clinical users can be granted access to the clinical Silver schema
when their role requires detailed records.

I keep these grants group-based so permissions remain manageable as users join
or leave the organization.

In [0]:
%sql
-- granting analytical users access to the curated Gold layer.

GRANT USE CATALOG
ON CATALOG health_insurance
TO `healthcare_analysts`;

GRANT USE SCHEMA
ON SCHEMA health_insurance.gold
TO `healthcare_analysts`;

GRANT SELECT
ON SCHEMA health_insurance.gold
TO `healthcare_analysts`;

In [0]:
%sql
-- granting privileged clinical users access to the curated and clinical layers.

GRANT USE CATALOG
ON CATALOG health_insurance
TO `healthcare_privileged`;

GRANT USE SCHEMA
ON SCHEMA health_insurance.gold
TO `healthcare_privileged`;

GRANT SELECT
ON SCHEMA health_insurance.gold
TO `healthcare_privileged`;

GRANT USE SCHEMA
ON SCHEMA health_insurance.silver
TO `healthcare_privileged`;

## Governance result

The Health Insurance lakehouse now applies multiple governance controls through
Unity Catalog.

### Data organization

I organize datasets through catalog and medallion schemas:

- Bronze
- Silver
- Gold
- Governance

### Data quality governance

Quality expectations are centrally stored in:

`health_insurance.governance.quality_rules`

The rules contain:

- dataset ownership
- severity
- rule version
- constraint definition
- source notebook lineage

### Sensitive-data classification

I classify direct Patient identifiers using the governed `pii` tag.

### Attribute-based access control

A schema-scoped ABAC policy automatically discovers columns tagged as PII and
masks them for analytical users.

This avoids maintaining separate masking logic for each sensitive column.

### Role-based access

I design permissions around account groups rather than individual users.

Analytical users consume curated Gold datasets.

Privileged clinical users can receive additional access to identifiable clinical
data when required.

### Design principle

I combine RBAC and ABAC:

RBAC controls whether a user can access a dataset.

ABAC controls what sensitive values the user can see within authorized
datasets.

This provides a scalable least-privilege governance model without embedding
personal user identities into the project configuration.